In [1]:
pip install vllm

In [2]:
pip install ray

  Using cached ray-2.55.1-cp312-cp312-manylinux2014_x86_64.whl.metadata (21 kB)
Using cached ray-2.55.1-cp312-cp312-manylinux2014_x86_64.whl (73.8 MB)


In [3]:
import re
import os
import sys
import ray
import io
import pandas as pd
from collections import Counter
from datasets import load_dataset
from datasets import get_dataset_split_names
from vllm import LLM, SamplingParams

In [4]:
os.environ["VLLM_NO_PRINTER_INITIALIZATION"] = "1"
print(f"VLLM_NO_PRINTER_INITIALIZATION set to: {os.environ.get('VLLM_NO_PRINTER_INITIALIZATION')}")

VLLM_NO_PRINTER_INITIALIZATION set to: 1


### Helper Functions

---



In [5]:
def parse_gsm8k_answer(generated_text: str):
    """
    Extract the final numeric answer from a model's CoT generation.
    Returns a float, or None if parsing fails.
    """
    # Step 1: cut off at the next "Q:" in case the model kept generating
    # past its own answer (common failure mode with few-shot prompting)
    text = generated_text.split("Q:")[0]

    # Step 2: find "The answer is" — case-insensitive, allow minor variants
    # Some models say "the answer is:", "answer: ", etc.
    match = re.search(
        r"(?:the answer is|answer is|answer:)\s*\$?\s*(-?[\d,]+(?:\.\d+)?)",
        text,
        re.IGNORECASE,
    )

    if match:
        num_str = match.group(1)
    else:
        # Fallback: grab the last number in the text
        # Works surprisingly well when the model forgets the template
        numbers = re.findall(r"-?[\d,]+(?:\.\d+)?", text)
        if not numbers:
            return None
        num_str = numbers[-1]

    # Step 3: clean and cast
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


def parse_gsm8k_gold(answer_field: str) -> float:
    # GSM8K gold answers look like: "...reasoning... #### 18"
    return float(answer_field.split("####")[-1].strip().replace(",", ""))



def is_correct(pred: float | None, gold: float) -> bool:
    if pred is None:
        return False
    return abs(pred - gold) < 1e-6  # float comparison tolerance

### End of Helper Functions

---





In [6]:
get_dataset_split_names("openai/gsm8k","main")

dataset = load_dataset("openai/gsm8k","main")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [7]:
exemplars = """Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done,
there will be 21 trees. How many trees did the grove workers plant today?
A: Westart with 15 trees. Later we have 21 trees. The difference must be the number of trees they planted.
So, they must have planted 21- 15 = 6 trees. The answer is 6.
Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: There are 3 cars in the parking lot already. 2 more arrive. Now there are 3 + 2 = 5 cars. The answer is 5.
Q: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
A: Leah had 32 chocolates and Leah’s sister had 42. That means there were originally 32 + 42 = 74
chocolates. 35 have been eaten. So in total they still have 74- 35 = 39 chocolates. The answer is 39.
Q: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops
did Jason give to Denny?
A: Jason had 20 lollipops. Since he only has 12 now, he must have given the rest to Denny. The number of
lollipops he has given to Denny must have been 20- 12 = 8 lollipops. The answer is 8.
Q: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does
he have now?
A: He has 5 toys. He got 2 from mom, so after that he has 5 + 2 = 7 toys. Then he got 2 more from dad, so
in total he has 7 + 2 = 9 toys. The answer is 9.
Q: There were nine computers in the server room. Five more computers were installed each day, from
monday to thursday. How many computers are now in the server room?
A: There are 4 days from monday to thursday. 5 computers were added each day. That means in total 4 * 5 =
20 computers were added. There were 9 computers in the beginning, so now there are 9 + 20 = 29 computers.
The answer is 29.
Q: Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many
golf balls did he have at the end of wednesday?
A: Michael initially had 58 balls. He lost 23 on Tuesday, so after that he has 58- 23 = 35 balls. On
Wednesday he lost 2 more so now he has 35- 2 = 33 balls. The answer is 33.
Q: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?
A: She bought 5 bagels for $3 each. This means she spent 5 * $3 = $15 on the bagels. She had $23 in
beginning, so now she has $23- $15 = $8. The answer is 8."""

In [8]:
# Restore original stdout if it was wrapped in a previous execution
if hasattr(sys.stdout, 'original'):
    sys.stdout = sys.stdout.original

class IOWrapper:
    def __init__(self, original):
        self.original = original
        # Open devnull to get a real, valid file descriptor
        self.devnull_fd = os.open(os.devnull, os.O_WRONLY)

    def fileno(self):
        # Return a valid file descriptor that os.dup can safely copy
        return self.devnull_fd

    def __getattr__(self, name):
        return getattr(self.original, name)

try:
    sys.stdout.fileno()
except io.UnsupportedOperation:
    sys.stdout = IOWrapper(sys.stdout)

ray.init(ignore_reinit_error=True)
llm = LLM(model="microsoft/Phi-3-mini-4k-instruct", enable_prefix_caching=False)
print("vLLM LLM model initialized successfully with devnull fileno patch.")


2026-05-09 09:28:00,632	INFO worker.py:2012 -- Started a local Ray instance.


INFO 05-09 09:28:04 [utils.py:233] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'model': 'microsoft/Phi-3-mini-4k-instruct'}
WARNING 05-09 09:28:04 [envs.py:1830] Unknown vLLM environment variable detected: VLLM_NO_PRINTER_INITIALIZATION


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

INFO 05-09 09:28:23 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-09 09:28:23 [nixl_utils.py:34] NIXL is not available
WARNING 05-09 09:28:23 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-09 09:28:23 [model.py:555] Resolved architecture: Phi3ForCausalLM
INFO 05-09 09:28:23 [model.py:1680] Using max model len 4096
INFO 05-09 09:28:23 [arg_utils.py:1711] Using ray runtime env (env vars redacted): {}
INFO 05-09 09:28:23 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-09 09:28:23 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-09 09:28:23 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


...

INFO 05-09 09:28:51 [weight_utils.py:615] Time spent downloading weights for microsoft/Phi-3-mini-4k-instruct: 20.636118 seconds
INFO 05-09 09:28:51 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.12 GiB. Available RAM: 76.45 GiB.
INFO 05-09 09:28:51 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-09 09:28:54 [default_loader.py:384] Loading weights took 2.20 seconds
INFO 05-09 09:28:55 [gpu_model_runner.py:4879] Model loading took 7.12 GiB memory and 25.230056 seconds
INFO 05-09 09:29:04 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/84fefe2475/rank_0_0/backbone for vLLM's torch.compile
INFO 05-09 09:29:04 [backends.py:1128] Dynamo bytecode transform time: 9.12 s
INFO 05-09 09:29:08 [backends.py:376] Cache the graph of compile range (1, 8192) for later use
INFO 05-09 09:29:13 [backends.py:391] Compiling a graph for compile range (1, 8192) takes 8.56 s
INFO 05-09 09:29:17 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/6a39a5053a717777303d8d2a41162d32a4d0b081be0a8aed9b5f84819d6db319/rank_0_0/model
INFO 05-09 09:29:17 [monitor.py:53] torch.compile took 22.40 s in total
INFO 05-09 09:29:18 [monitor.py:81] Initial profiling/warmup run took 0.83 s
INFO 05-09 09:29:28 [gpu_model_runner.

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 20.63it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 25.13it/s]


INFO 05-09 09:29:36 [gpu_model_runner.py:6133] Graph capturing finished in 5 secs, took 0.47 GiB
INFO 05-09 09:29:36 [gpu_worker.py:599] CUDA graph pool memory: 0.47 GiB (actual), 0.51 GiB (estimated), difference: 0.04 GiB (7.9%).
INFO 05-09 09:29:36 [core.py:299] init engine (profile, create kv cache, warmup model) took 41.40 s (compilation: 22.40 s)
INFO 05-09 09:29:37 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
vLLM LLM model initialized successfully with devnull fileno patch.


In [9]:
dataset['test'].num_rows

1319

***Self-Consistency (for Phi-3-mini)***

In [10]:
sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=500,
    n=40,                    # 40 samples per prompt
    stop=["\nQ:"],           # stop at next question
)

Number_of_Questions_to_Tested =  50

sc_question_answer_dict = {}
for i in range(0,dataset['test'].num_rows):
   question = dataset["test"][i]["question"]
   full_prompt = exemplars + f"\n\nQ: {question}\nA:"

   outputs = llm.generate(full_prompt, sampling_params)

   for req in outputs:
    print("request_id:", req.request_id)
    print("prompt:", req.prompt)

    answers = []

    for comp in req.outputs:
        print("completion index:", comp.index)
        print("text:", comp.text)
        print("finish_reason:", comp.finish_reason)
        answers.append(parse_gsm8k_answer(comp.text))

    final_answer = Counter(a for a in answers if a is not None).most_common(1)[0][0]

    sc_question_answer_dict[i] = [question,final_answer,parse_gsm8k_gold(dataset["test"][i]["answer"]),is_correct(final_answer,parse_gsm8k_gold(dataset["test"][i]["answer"]))]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/40 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

request_id: 0
prompt: Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done,
there will be 21 trees. How many trees did the grove workers plant today?
A: Westart with 15 trees. Later we have 21 trees. The difference must be the number of trees they planted.
So, they must have planted 21- 15 = 6 trees. The answer is 6.
Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: There are 3 cars in the parking lot already. 2 more arrive. Now there are 3 + 2 = 5 cars. The answer is 5.
Q: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
A: Leah had 32 chocolates and Leah’s sister had 42. That means there were originally 32 + 42 = 74
chocolates. 35 have been eaten. So in total they still have 74- 35 = 39 chocolates. The answer is 39.
Q: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/40 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/40 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

request_id: 37
prompt: Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done,
there will be 21 trees. How many trees did the grove workers plant today?
A: Westart with 15 trees. Later we have 21 trees. The difference must be the number of trees they planted.
So, they must have planted 21- 15 = 6 trees. The answer is 6.
Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: There are 3 cars in the parking lot already. 2 more arrive. Now there are 3 + 2 = 5 cars. The answer is 5.
Q: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
A: Leah had 32 chocolates and Leah’s sister had 42. That means there were originally 32 + 42 = 74
chocolates. 35 have been eaten. So in total they still have 74- 35 = 39 chocolates. The answer is 39.
Q: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/40 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

KeyboardInterrupt: 

In [ ]:
sc_question_answer_dict

In [ ]:
df = pd.DataFrame.from_dict(sc_question_answer_dict, orient='index')

In [ ]:
df

,0,1,2,3
0,Janet’s ducks lay 16 eggs per day. She eats th...,18.0,18.0,True
1,A robe takes 2 bolts of blue fiber and half th...,3.0,3.0,True
2,Josh decides to try flipping a house. He buys...,70000.0,70000.0,True
3,James decides to run 3 sprints 3 times a week....,540.0,540.0,True
4,"Every day, Wendi feeds each of her chickens th...",20.0,20.0,True
...,...,...,...,...
272,A mother goes shopping. She buys cocoa at $4.2...,5.0,5.0,True
273,"Terri is knitting a sweater with two sleeves, ...",315.0,315.0,True
274,"It's April, and Mrs. Rylan has been busy on he...",3200.0,3200.0,True
275,Sean is practicing for his role in a theater p...,138.0,138.0,True


In [ ]:
df.to_excel("results_0_276.xlsx")

In [ ]:
# Destroy the old dead engine

import gc
gc.collect()
import torch
torch.cuda.empty_cache()

# Recreate it
llm = LLM(model="microsoft/Phi-3-mini-4k-instruct", gpu_memory_utilization=0.9)

INFO 04-22 06:36:40 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'microsoft/Phi-3-mini-4k-instruct'}
WARNING 04-22 06:36:40 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_NO_PRINTER_INITIALIZATION


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

INFO 04-22 06:36:53 [model.py:549] Resolved architecture: Phi3ForCausalLM
INFO 04-22 06:36:53 [model.py:1678] Using max model len 4096
INFO 04-22 06:36:53 [arg_utils.py:1611] Using ray runtime env (env vars redacted): {}


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

INFO 04-22 06:37:18 [weight_utils.py:581] Time spent downloading weights for microsoft/Phi-3-mini-4k-instruct: 18.321961 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-22 06:37:21 [default_loader.py:384] Loading weights took 2.23 seconds
INFO 04-22 06:37:22 [gpu_model_runner.py:4820] Model loading took 7.12 GiB memory and 22.106549 seconds
INFO 04-22 06:37:31 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/0fb7bdd23e/rank_0_0/backbone for vLLM's torch.compile
INFO 04-22 06:37:31 [backends.py:1111] Dynamo bytecode transform time: 7.78 s
INFO 04-22 06:37:33 [backends.py:372] Cache the graph of compile range (1, 8192) for later use
INFO 04-22 06:37:39 [backends.py:390] Compiling a graph for compile range (1, 8192) takes 8.20 s
INFO 04-22 06:37:41 [decorators.py:655] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/94e30a0c092cb90ae9a983f8de6279ab4f775e554109d9a08245b256c03455b7/rank_0_0/model
INFO 04-22 06:37:41 [monitor.py:48] torch.compile took 18.10 s in total
INFO 04-22 06:37:42 [monitor.py:76] Initial profiling/warmup run took 0.83 s
INFO 04-22 06:37:52 [kv_cache_utils.py

(EngineCore pid=5802) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   0%|          | 0/51 [00:00<?, ?it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   4%|▍         | 2/51 [00:00<00:02, 18.29it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   8%|▊         | 4/51 [00:00<00:02, 17.56it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  12%|█▏        | 6/51 [00:00<00:02, 17.62it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  16%|█▌        | 8/51 [00:00<00:02, 18.06it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  22%|██▏       | 11/51 [00:00<00:02, 19.54it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  27%|██▋       | 14/51 [00:00<00:01, 19.83it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  33%|███▎      | 17/51 [00:00<00:01, 20.27it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  39%|███▉      | 20/51 [00:01<00:01, 20.74it/s]
Capturing CUDA graphs (mixed prefill-decode, 

INFO 04-22 06:38:01 [gpu_model_runner.py:6046] Graph capturing finished in 6 secs, took 0.47 GiB
INFO 04-22 06:38:01 [gpu_worker.py:597] CUDA graph pool memory: 0.47 GiB (actual), 0.51 GiB (estimated), difference: 0.04 GiB (7.9%).
INFO 04-22 06:38:01 [core.py:283] init engine (profile, create kv cache, warmup model) took 38.97 seconds


In [ ]:
sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=500,
    n=40,
    stop=["\nQ:"],
)

# Step 1: Build ALL prompts at once
start_idx = 277
all_prompts = []
question_indices = []

for i in range(start_idx, dataset['test'].num_rows):
    question = dataset["test"][i]["question"]
    full_prompt = exemplars + f"\n\nQ: {question}\nA:"
    all_prompts.append(full_prompt)
    question_indices.append(i)

# Step 2: One single vLLM call — it batches everything internally
all_outputs = llm.generate(all_prompts, sampling_params)

# Step 3: Process all results
sc_question_answer_dict_batch2 = {}

for idx, output in enumerate(all_outputs):
    real_index = question_indices[idx]
    question = dataset["test"][real_index]["question"]

    answers = []
    for comp in output.outputs:
        answers.append(parse_gsm8k_answer(comp.text))

    final_answer = Counter(a for a in answers if a is not None).most_common(1)[0][0]
    gold = parse_gsm8k_gold(dataset["test"][real_index]["answer"])

    sc_question_answer_dict_batch2[real_index] = [
        question,
        final_answer,
        gold,
        is_correct(final_answer, gold),
        answers,  # store raw 40 samples for later analysis
    ]

# Step 4: Save immediately
import json
with open("checkpoint_277_1318.json", "w") as f:
    json.dump(sc_question_answer_dict_batch2, f)

Rendering prompts:   0%|          | 0/1042 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
df2 = pd.DataFrame.from_dict(sc_question_answer_dict_batch2, orient='index')

In [ ]:
df2.to_excel("results_277_1318.xlsx")

***Greedy Baseline(for Phi-3-mini)***

In [ ]:
import gc
gc.collect()
import torch
torch.cuda.empty_cache()

# Recreate it
#llm = LLM(model="microsoft/Phi-3-mini-4k-instruct", gpu_memory_utilization=0.9)

In [ ]:
greedy_params = SamplingParams(
    temperature=0,
    max_tokens=500,
    n=1,
    stop=["\nQ:"],
)


all_prompts = [
    exemplars + f"\n\nQ: {dataset['test'][i]['question']}\nA:"
    for i in range(dataset['test'].num_rows)
]


greedy_outputs = llm.generate(all_prompts, greedy_params)

greedy_question_answer_dict_batch2 = {}

for idx, output in enumerate(greedy_outputs):
    question = dataset["test"][idx]["question"]

    answers = []
    for comp in output.outputs:
        answers.append(parse_gsm8k_answer(comp.text))

    final_answer = answers[0]
    gold = parse_gsm8k_gold(dataset["test"][idx]["answer"])

    greedy_question_answer_dict_batch2[idx] = [
        question,
        final_answer,
        gold,
        is_correct(final_answer, gold),
        answers,  # store raw 40 samples for later analysis
    ]

# Step 4: Save immediately
import json
with open("greedy_0_1318.json", "w") as f:
    json.dump(greedy_question_answer_dict_batch2, f)

Rendering prompts:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

In [ ]:
greedy_question_answer_dict_batch2

{0: ["Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  18.0,
  18.0,
  True,
  [18.0]],
 1: ['A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?',
  3.0,
  3.0,
  True,
  [3.0]],
 2: ['Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?',
  70000.0,
  70000.0,
  True,
  [70000.0]],
 3: ['James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?',
  540.0,
  540.0,
  True,
  [540.0]],
 4: ["Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables t

In [ ]:
df3 = pd.DataFrame.from_dict(greedy_question_answer_dict_batch2, orient='index')
df3.to_excel("results_0_1318.xlsx")

**Mistral-7B**

In [ ]:
# Restore original stdout if it was wrapped in a previous execution
if hasattr(sys.stdout, 'original'):
    sys.stdout = sys.stdout.original

class IOWrapper:
    def __init__(self, original):
        self.original = original
        # Open devnull to get a real, valid file descriptor
        self.devnull_fd = os.open(os.devnull, os.O_WRONLY)

    def fileno(self):
        # Return a valid file descriptor that os.dup can safely copy
        return self.devnull_fd

    def __getattr__(self, name):
        return getattr(self.original, name)

try:
    sys.stdout.fileno()
except io.UnsupportedOperation:
    sys.stdout = IOWrapper(sys.stdout)

ray.init(ignore_reinit_error=True)
llm = LLM(model="mistralai/Mistral-7B-Instruct-v0.3", enable_prefix_caching=False)
print("vLLM LLM model initialized successfully with devnull fileno patch.")


2026-04-22 06:38:37,299	INFO worker.py:1828 -- Calling ray.init() again after it has already been called.


INFO 04-22 06:38:37 [utils.py:233] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
WARNING 04-22 06:38:37 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_NO_PRINTER_INITIALIZATION
INFO 04-22 06:38:37 [model.py:549] Resolved architecture: MistralForCausalLM
INFO 04-22 06:38:37 [model.py:1678] Using max model len 32768
INFO 04-22 06:38:37 [arg_utils.py:1611] Using ray runtime env (env vars redacted): {}
INFO 04-22 06:38:37 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='mistralai/Mistral-7B-Instruct-v0.3', speculative_config=None, tokenizer='mistralai/Mistral-7B-Instruct-v0.3', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_

(EngineCore pid=30522) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=30522) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


INFO 04-22 06:38:41 [weight_utils.py:625] No consolidated.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 04-22 06:38:46 [default_loader.py:384] Loading weights took 4.79 seconds
INFO 04-22 06:38:47 [gpu_model_runner.py:4820] Model loading took 13.51 GiB memory and 6.410586 seconds
INFO 04-22 06:38:51 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/28b564c5f4/rank_0_0/backbone for vLLM's torch.compile
INFO 04-22 06:38:51 [backends.py:1111] Dynamo bytecode transform time: 4.16 s
INFO 04-22 06:38:56 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 4.508 s
INFO 04-22 06:38:56 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/9da8eeaac3e68465d6b4ae4b4f6132da3c4280eda4546048a88e333943544021/rank_0_0/model
INFO 04-22 06:38:56 [monitor.py:48] torch.compile took 9.01 s in total
INFO 04-22 06:38:56 [monitor.py:76] Initial profiling/warmup run took 0.22 s
INFO 04-22 06:38:58 [kv_cache_utils.py:829] Overriding num_gpu_blocks=0 with num_gpu_blocks

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 18.36it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 21.77it/s]


INFO 04-22 06:39:06 [gpu_model_runner.py:6046] Graph capturing finished in 5 secs, took 0.48 GiB
INFO 04-22 06:39:06 [gpu_worker.py:597] CUDA graph pool memory: 0.48 GiB (actual), 0.53 GiB (estimated), difference: 0.04 GiB (8.9%).
INFO 04-22 06:39:06 [core.py:283] init engine (profile, create kv cache, warmup model) took 18.85 seconds
vLLM LLM model initialized successfully with devnull fileno patch.


***Self-Consistency Computation(for Mistral-7B)***

In [ ]:
sc_sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=500,
    n=40,                    # 40 samples per prompt
    stop=["\nQ:"],           # stop at next question
)


all_prompts = [
    exemplars + f"\n\nQ: {dataset['test'][i]['question']}\nA:"
    for i in range(dataset['test'].num_rows)
]


sc_mistral_outputs = llm.generate(all_prompts, sc_sampling_params)

sc_question_answer_dict_batch = {}

for idx, output in enumerate(sc_mistral_outputs):
    question = dataset["test"][idx]["question"]

    answers = []
    for comp in output.outputs:
        answers.append(parse_gsm8k_answer(comp.text))

    final_answer = Counter(a for a in answers if a is not None).most_common(1)[0][0]
    gold = parse_gsm8k_gold(dataset["test"][idx]["answer"])

    sc_question_answer_dict_batch[idx] = [
        question,
        final_answer,
        gold,
        is_correct(final_answer, gold),
        answers,  # store raw 40 samples for later analysis
    ]

# Step 4: Save immediately
import json
with open("sc_mistral_0_1318.json", "w") as f:
    json.dump(sc_question_answer_dict_batch, f)

Rendering prompts:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/52760 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

***Greedy Baseline(for Mistral-7B)***

In [ ]:
greedy_params = SamplingParams(
    temperature=0,
    max_tokens=500,
    n=1,
    stop=["\nQ:"],
)


all_prompts = [
    exemplars + f"\n\nQ: {dataset['test'][i]['question']}\nA:"
    for i in range(dataset['test'].num_rows)
]


greedy_outputs = llm.generate(all_prompts, greedy_params)

greedy_question_answer_dict_batch2 = {}

for idx, output in enumerate(greedy_outputs):
    question = dataset["test"][idx]["question"]

    answers = []
    for comp in output.outputs:
        answers.append(parse_gsm8k_answer(comp.text))

    final_answer = answers[0]
    gold = parse_gsm8k_gold(dataset["test"][idx]["answer"])

    greedy_question_answer_dict_batch2[idx] = [
        question,
        final_answer,
        gold,
        is_correct(final_answer, gold),
        answers,  # store raw 40 samples for later analysis
    ]

# Step 4: Save immediately
import json
with open("greedy_mistral_0_1318.json", "w") as f:
    json.dump(greedy_question_answer_dict_batch2, f)

Rendering prompts:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

In [ ]:
greedy_correct = sum(1 for v in greedy_question_answer_dict_batch2.values() if v[3])
sc_correct = sum(1 for v in sc_question_answer_dict_batch.values() if v[3])
total = len(greedy_question_answer_dict_batch2)

greedy_acc = greedy_correct / total * 100
sc_acc = sc_correct / total * 100

print(f"Model: Mistral-7B-Instruct-v0.3")
print(f"Greedy:            {greedy_acc:.1f}% ({greedy_correct}/{total})")
print(f"Self-Consistency:  {sc_acc:.1f}% ({sc_correct}/{total})")
print(f"SC Gain:           +{sc_acc - greedy_acc:.1f}%")

Model: Mistral-7B-Instruct-v0.3
Greedy:            54.0% (712/1319)
Self-Consistency:  72.9% (962/1319)
SC Gain:           +19.0%


In [ ]:
print(f"{'Model':<30} {'Greedy':>8} {'SC':>8} {'Gain':>8}")
print("-" * 56)
print(f"{'Phi-3-mini (3.8B)':<30} {'84.8%':>8} {'91.3%':>8} {'+6.4%':>8}")
print(f"{'Mistral-7B (7.2B)':<30} {f'{greedy_acc:.1f}%':>8} {f'{sc_acc:.1f}%':>8} {f'+{sc_acc-greedy_acc:.1f}%':>8}")

Model                            Greedy       SC     Gain
--------------------------------------------------------
Phi-3-mini (3.8B)                 84.8%    91.3%    +6.4%
Mistral-7B (7.2B)                 54.0%    72.9%   +19.0%


Gemma-2 9B

In [ ]:
# Restore original stdout if it was wrapped in a previous execution
if hasattr(sys.stdout, 'original'):
    sys.stdout = sys.stdout.original

class IOWrapper:
    def __init__(self, original):
        self.original = original
        # Open devnull to get a real, valid file descriptor
        self.devnull_fd = os.open(os.devnull, os.O_WRONLY)

    def fileno(self):
        # Return a valid file descriptor that os.dup can safely copy
        return self.devnull_fd

    def __getattr__(self, name):
        return getattr(self.original, name)

try:
    sys.stdout.fileno()
except io.UnsupportedOperation:
    sys.stdout = IOWrapper(sys.stdout)

ray.init(ignore_reinit_error=True)
llm = LLM(model="google/gemma-2-9b-it", enable_prefix_caching=False)
print("vLLM LLM model initialized successfully with devnull fileno patch.")


2026-04-30 05:59:01,071	INFO worker.py:2012 -- Started a local Ray instance.


INFO 04-30 05:59:04 [utils.py:233] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'model': 'google/gemma-2-9b-it'}
WARNING 04-30 05:59:04 [envs.py:1818] Unknown vLLM environment variable detected: VLLM_NO_PRINTER_INITIALIZATION


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

INFO 04-30 05:59:22 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 04-30 05:59:22 [nixl_utils.py:34] NIXL is not available
WARNING 04-30 05:59:22 [nixl_utils.py:44] NIXL agent config is not available
INFO 04-30 05:59:22 [model.py:555] Resolved architecture: Gemma2ForCausalLM
INFO 04-30 05:59:22 [model.py:1680] Using max model len 8192
INFO 04-30 05:59:22 [arg_utils.py:1711] Using ray runtime env (env vars redacted): {}
INFO 04-30 05:59:22 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-30 05:59:22 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 04-30 05:59:22 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


...

INFO 04-30 06:00:16 [weight_utils.py:615] Time spent downloading weights for google/gemma-2-9b-it: 46.970642 seconds
INFO 04-30 06:00:16 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 17.21 GiB. Available RAM: 76.26 GiB.
INFO 04-30 06:00:16 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 04-30 06:00:23 [default_loader.py:384] Loading weights took 6.41 seconds
INFO 04-30 06:00:23 [gpu_model_runner.py:4879] Model loading took 17.22 GiB memory and 55.108828 seconds
INFO 04-30 06:00:36 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/e38b103181/rank_0_0/backbone for vLLM's torch.compile
INFO 04-30 06:00:36 [backends.py:1128] Dynamo bytecode transform time: 12.10 s
INFO 04-30 06:00:42 [backends.py:376] Cache the graph of compile range (1, 8192) for later use
INFO 04-30 06:00:49 [backends.py:391] Compiling a graph for compile range (1, 8192) takes 12.03 s
INFO 04-30 06:00:55 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/39b04c9d8c1b528a1bb848380242aa31e3afe7ae5e812770612fd3223b98bd3e/rank_0_0/model
INFO 04-30 06:00:55 [monitor.py:53] torch.compile took 30.85 s in total
INFO 04-30 06:00:56 [monitor.py:81] Initial profiling/warmup run took 1.06 s
INFO 04-30 06:01:06 [gpu_model_runn

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 13.63it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 17.29it/s]


INFO 04-30 06:01:15 [gpu_model_runner.py:6133] Graph capturing finished in 7 secs, took 0.56 GiB
INFO 04-30 06:01:15 [gpu_worker.py:599] CUDA graph pool memory: 0.56 GiB (actual), 0.63 GiB (estimated), difference: 0.07 GiB (11.8%).
INFO 04-30 06:01:15 [core.py:299] init engine (profile, create kv cache, warmup model) took 52.03 s (compilation: 30.85 s)
INFO 04-30 06:01:16 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
vLLM LLM model initialized successfully with devnull fileno patch.


Self-Consistency Computation (for Gemma-9B)

In [ ]:
sc_sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=500,
    n=40,                    # 40 samples per prompt
    stop=["\nQ:"],           # stop at next question
)


all_prompts = [
    exemplars + f"\n\nQ: {dataset['test'][i]['question']}\nA:"
    for i in range(dataset['test'].num_rows)
]


sc_gemma_outputs = llm.generate(all_prompts, sc_sampling_params)

sc_question_answer_dict_batch = {}

for idx, output in enumerate(sc_gemma_outputs):
    question = dataset["test"][idx]["question"]

    answers = []
    for comp in output.outputs:
        answers.append(parse_gsm8k_answer(comp.text))

    final_answer = Counter(a for a in answers if a is not None).most_common(1)[0][0]
    gold = parse_gsm8k_gold(dataset["test"][idx]["answer"])

    sc_question_answer_dict_batch[idx] = [
        question,
        final_answer,
        gold,
        is_correct(final_answer, gold),
        answers,  # store raw 40 samples for later analysis
    ]

# Step 4: Save immediately
import json
with open("sc_gemma_0_1318.json", "w") as f:
    json.dump(sc_question_answer_dict_batch, f)

Rendering prompts:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/52760 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

***Greedy Baseline Computation (for Gemma-9B)***

In [ ]:
greedy_params = SamplingParams(
    temperature=0,
    max_tokens=500,
    n=1,
    stop=["\nQ:"],
)


all_prompts = [
    exemplars + f"\n\nQ: {dataset['test'][i]['question']}\nA:"
    for i in range(dataset['test'].num_rows)
]


greedy_outputs = llm.generate(all_prompts, greedy_params)

greedy_question_answer_dict_batch2 = {}

for idx, output in enumerate(greedy_outputs):
    question = dataset["test"][idx]["question"]

    answers = []
    for comp in output.outputs:
        answers.append(parse_gsm8k_answer(comp.text))

    final_answer = answers[0]
    gold = parse_gsm8k_gold(dataset["test"][idx]["answer"])

    greedy_question_answer_dict_batch2[idx] = [
        question,
        final_answer,
        gold,
        is_correct(final_answer, gold),
        answers,  # store raw 40 samples for later analysis
    ]

# Step 4: Save immediately
import json
with open("greedy_gemma_0_1318.json", "w") as f:
    json.dump(greedy_question_answer_dict_batch2, f)

Rendering prompts:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

In [ ]:
import json

In [ ]:
with  open('/content/greedy_gemma_0_1318.json') as json_file:
  greedy_question_answer_dict_gemma = json.load(json_file)


In [ ]:
with  open('/content/sc_gemma_0_1318.json') as json_file:
  sc_question_answer_dict_gemma = json.load(json_file)


In [ ]:
greedy_correct_gemma = sum(1 for v in greedy_question_answer_dict_gemma.values() if v[3])
sc_correct_gemma = sum(1 for v in sc_question_answer_dict_gemma.values() if v[3])
total = len(greedy_question_answer_dict_gemma)

greedy_acc_gemma = greedy_correct_gemma / total * 100
sc_acc_gemma = sc_correct_gemma / total * 100

print(f"Model: Gemma-9B-it")
print(f"Greedy:            {greedy_acc_gemma:.1f}% ({greedy_correct_gemma}/{total})")
print(f"Self-Consistency:  {sc_acc_gemma:.1f}% ({sc_correct_gemma}/{total})")
print(f"SC Gain:           +{sc_acc_gemma - greedy_acc_gemma:.1f}%")

Model: Gemma-9B-it
Greedy:            86.1% (1136/1319)
Self-Consistency:  89.4% (1179/1319)
SC Gain:           +3.3%


In [ ]:
greedy_correct_gemma

1136

In [ ]:
print(f"{'Model':<30} {'Greedy':>8} {'SC':>8} {'Gain':>8}")
print("-" * 56)
print(f"{'Phi-3-mini (3.8B)':<30} {'84.8%':>8} {'91.3%':>8} {'+6.4%':>8}")
print(f"{'Gemma-9B-it':<30} {f'{greedy_acc:.1f}%':>8} {f'{sc_acc:.1f}%':>8} {f'+{sc_acc-greedy_acc:.1f}%':>8}")

Model                            Greedy       SC     Gain
--------------------------------------------------------
Phi-3-mini (3.8B)                 84.8%    91.3%    +6.4%
Gemma-9B-it                       86.1%    89.4%    +3.3%


Gemma - 2b -it

In [ ]:
# Restore original stdout if it was wrapped in a previous execution
if hasattr(sys.stdout, 'original'):
    sys.stdout = sys.stdout.original

class IOWrapper:
    def __init__(self, original):
        self.original = original
        # Open devnull to get a real, valid file descriptor
        self.devnull_fd = os.open(os.devnull, os.O_WRONLY)

    def fileno(self):
        # Return a valid file descriptor that os.dup can safely copy
        return self.devnull_fd

    def __getattr__(self, name):
        return getattr(self.original, name)

try:
    sys.stdout.fileno()
except io.UnsupportedOperation:
    sys.stdout = IOWrapper(sys.stdout)

ray.init(ignore_reinit_error=True)
llm = LLM(model="google/gemma-2-2b-it", enable_prefix_caching=False)
print("vLLM LLM model initialized successfully with devnull fileno patch.")

2026-05-01 12:40:04,402	INFO worker.py:1828 -- Calling ray.init() again after it has already been called.


INFO 05-01 12:40:04 [utils.py:233] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'model': 'google/gemma-2-2b-it'}
WARNING 05-01 12:40:04 [envs.py:1818] Unknown vLLM environment variable detected: VLLM_NO_PRINTER_INITIALIZATION
INFO 05-01 12:40:05 [model.py:555] Resolved architecture: Gemma2ForCausalLM
INFO 05-01 12:40:05 [model.py:1680] Using max model len 8192
INFO 05-01 12:40:05 [arg_utils.py:1711] Using ray runtime env (env vars redacted): {}
INFO 05-01 12:40:05 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
INFO 05-01 12:40:05 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='google/gemma-2-2b-it', speculative_config=None, tokenizer='google/gemma-2-2b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1,

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-01 12:40:12 [default_loader.py:384] Loading weights took 1.71 seconds
INFO 05-01 12:40:13 [gpu_model_runner.py:4879] Model loading took 4.9 GiB memory and 4.145772 seconds
INFO 05-01 12:40:17 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/1bb6eb19f9/rank_0_0/backbone for vLLM's torch.compile
INFO 05-01 12:40:17 [backends.py:1128] Dynamo bytecode transform time: 3.51 s
INFO 05-01 12:40:19 [backends.py:290] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.738 s
INFO 05-01 12:40:19 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/8f4a6c1f1b74e25dd3b989cbb77677f3a700a8d21108b9223b6f586c929a43d0/rank_0_0/model
INFO 05-01 12:40:19 [monitor.py:53] torch.compile took 5.56 s in total
INFO 05-01 12:40:19 [monitor.py:81] Initial profiling/warmup run took 0.66 s
INFO 05-01 12:40:20 [gpu_model_runner.py:5963] Profiling CUDA graph memory: PIECEWISE=51 (lar

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 25.87it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 29.67it/s]


INFO 05-01 12:40:26 [gpu_model_runner.py:6133] Graph capturing finished in 4 secs, took 0.43 GiB
INFO 05-01 12:40:26 [gpu_worker.py:599] CUDA graph pool memory: 0.43 GiB (actual), 0.41 GiB (estimated), difference: 0.02 GiB (3.7%).
INFO 05-01 12:40:26 [core.py:299] init engine (profile, create kv cache, warmup model) took 13.93 s (compilation: 5.56 s)
INFO 05-01 12:40:27 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
vLLM LLM model initialized successfully with devnull fileno patch.


***Self-Consistency Computation (for Gemma-2B)***

In [ ]:
sc_sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=500,
    n=40,                    # 40 samples per prompt
    stop=["\nQ:"],           # stop at next question
)


all_prompts = [
    exemplars + f"\n\nQ: {dataset['test'][i]['question']}\nA:"
    for i in range(dataset['test'].num_rows)
]


sc_gemma_2billion_outputs = llm.generate(all_prompts, sc_sampling_params)

sc_question_answer_dict_batch = {}

for idx, output in enumerate(sc_gemma_2billion_outputs):
    question = dataset["test"][idx]["question"]

    answers = []
    for comp in output.outputs:
        answers.append(parse_gsm8k_answer(comp.text))

    final_answer = Counter(a for a in answers if a is not None).most_common(1)[0][0]
    gold = parse_gsm8k_gold(dataset["test"][idx]["answer"])

    sc_question_answer_dict_batch[idx] = [
        question,
        final_answer,
        gold,
        is_correct(final_answer, gold),
        answers,  # store raw 40 samples for later analysis
    ]

# Step 4: Save immediately
import json
with open("sc_gemma_2_billion_0_1318.json", "w") as f:
    json.dump(sc_question_answer_dict_batch, f)

Rendering prompts:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/52760 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

Greedy Baseline Computation (for Gemma-2B)

In [ ]:
greedy_params = SamplingParams(
    temperature=0,
    max_tokens=500,
    n=1,
    stop=["\nQ:"],
)


all_prompts = [
    exemplars + f"\n\nQ: {dataset['test'][i]['question']}\nA:"
    for i in range(dataset['test'].num_rows)
]


greedy_outputs = llm.generate(all_prompts, greedy_params)

greedy_question_answer_dict_batch2 = {}

for idx, output in enumerate(greedy_outputs):
    question = dataset["test"][idx]["question"]

    answers = []
    for comp in output.outputs:
        answers.append(parse_gsm8k_answer(comp.text))

    final_answer = answers[0]
    gold = parse_gsm8k_gold(dataset["test"][idx]["answer"])

    greedy_question_answer_dict_batch2[idx] = [
        question,
        final_answer,
        gold,
        is_correct(final_answer, gold),
        answers,  # store raw 40 samples for later analysis
    ]

# Step 4: Save immediately
import json
with open("greedy_gemma_2billion_0_1318.json", "w") as f:
    json.dump(greedy_question_answer_dict_batch2, f)

Rendering prompts:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

In [ ]:
greedy_correct_gemma = sum(1 for v in greedy_question_answer_dict_batch2.values() if v[3])
sc_correct_gemma = sum(1 for v in sc_question_answer_dict_batch.values() if v[3])
total = len(greedy_question_answer_dict_batch2)

greedy_acc_gemma = greedy_correct_gemma / total * 100
sc_acc_gemma = sc_correct_gemma / total * 100

print(f"Model: Gemma-2B-it")
print(f"Greedy:            {greedy_acc_gemma:.1f}% ({greedy_correct_gemma}/{total})")
print(f"Self-Consistency:  {sc_acc_gemma:.1f}% ({sc_correct_gemma}/{total})")
print(f"SC Gain:           +{sc_acc_gemma - greedy_acc_gemma:.1f}%")

Model: Gemma-2B-it
Greedy:            50.2% (662/1319)
Self-Consistency:  59.5% (785/1319)
SC Gain:           +9.3%
